# HuggingFace Model Exploration & Comparison
*Research Stage: Understanding the HF Ecosystem*

This notebook helps you explore and compare different HuggingFace models to understand their capabilities, performance, and use cases.

## Learning Objectives
- Load and test different model types (GPT, BERT, T5, etc.)
- Compare model performance on various tasks
- Understand model size vs quality trade-offs
- Create an interactive model testing interface

## Prerequisites
- HuggingFace account and API token
- GPU recommended for larger models
- Basic understanding of transformers

## Setup and Imports

In [ ]:
# Install required packages (run this in a terminal if needed)
# pip install transformers torch accelerate huggingface_hub

import torch
from transformers import (
    pipeline, 
    AutoTokenizer, 
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification
)
import pandas as pd
import time
from huggingface_hub import HfApi
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Model Categories to Explore

Let's define different model categories and some popular examples from each.

In [ ]:
# Define model categories and examples
MODEL_CATEGORIES = {
    "text_generation": {
        "description": "Models for generating text (GPT-style)",
        "models": {
            "gpt2": "gpt2",
            "distilgpt2": "distilgpt2", 
            "microsoft/DialoGPT-small": "microsoft/DialoGPT-small",
            "microsoft/phi-1_5": "microsoft/phi-1_5"
        }
    },
    "text_to_text": {
        "description": "Sequence-to-sequence models (T5, FLAN-T5)",
        "models": {
            "t5-small": "t5-small",
            "google/flan-t5-small": "google/flan-t5-small",
            "facebook/blenderbot-400M-distill": "facebook/blenderbot-400M-distill"
        }
    },
    "sentiment_analysis": {
        "description": "Models for sentiment and emotion analysis",
        "models": {
            "distilbert-base-uncased-finetuned-sst-2-english": "distilbert-base-uncased-finetuned-sst-2-english",
            "cardiffnlp/twitter-roberta-base-sentiment": "cardiffnlp/twitter-roberta-base-sentiment"
        }
    }
}

# Display available models
for category, info in MODEL_CATEGORIES.items():
    print(f"\n🔍 {category.upper()}: {info['description']}")
    for model_name, model_id in info['models'].items():
        print(f"  • {model_name} → {model_id}")

## Interactive Model Tester

Create a function to test any model with custom prompts.

In [ ]:
def test_model(model_id, task_type="text-generation", test_prompt="Hello, how are you today?", **kwargs):
    """
    Test a HuggingFace model with a given prompt.
    
    Args:
        model_id: HuggingFace model ID
        task_type: Type of task (text-generation, text2text-generation, sentiment-analysis)
        test_prompt: Input prompt to test
        **kwargs: Additional pipeline arguments
    """
    print(f"\n🧪 Testing model: {model_id}")
    print(f"Task: {task_type}")
    print(f"Prompt: '{test_prompt}'")
    print("-" * 50)
    
    start_time = time.time()
    
    try:
        # Create pipeline
        if task_type == "text2text-generation":
            pipe = pipeline("text2text-generation", model=model_id, **kwargs)
        else:
            pipe = pipeline(task_type, model=model_id, **kwargs)
        
        # Run inference
        if task_type == "sentiment-analysis":
            result = pipe(test_prompt)
            print(f"Sentiment: {result[0]['label']} (confidence: {result[0]['score']:.3f})")
        else:
            result = pipe(test_prompt, max_length=50, num_return_sequences=1, truncation=True)
            if isinstance(result, list) and result:
                output = result[0]['generated_text'] if 'generated_text' in result[0] else result[0]
                print(f"Output: {output}")
            else:
                print(f"Output: {result}")
        
        inference_time = time.time() - start_time
        print(".2f")
        
        # Get model size info
        try:
            api = HfApi()
            model_info = api.model_info(model_id)
            if hasattr(model_info, 'siblings'):
                total_size = sum(sibling.size for sibling in model_info.siblings if sibling.size) / (1024**3)  # GB
                print(".2f")
        except:
            print("Model size: Unable to retrieve")
            
        return True
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return False

# Test the function
print("Testing the model tester function...")
test_model("gpt2", test_prompt="The future of AI is")

## Comparative Analysis

Let's compare models across different categories with the same prompt.

In [ ]:
def compare_models(test_prompt="Explain quantum computing in simple terms", categories_to_test=None):
    """
    Compare models from different categories on the same prompt.
    """
    if categories_to_test is None:
        categories_to_test = ["text_generation", "text_to_text"]
    
    results = []
    
    for category in categories_to_test:
        if category not in MODEL_CATEGORIES:
            continue
            
        print(f"\n🎯 Testing {category.upper()} models")
        print("=" * 60)
        
        for model_name, model_id in MODEL_CATEGORIES[category]['models'].items():
            success = test_model(
                model_id, 
                task_type=category if category != "text_generation" else "text-generation",
                test_prompt=test_prompt,
                device=0 if torch.cuda.is_available() else -1  # Use GPU if available
            )
            
            if success:
                results.append({
                    'category': category,
                    'model_name': model_name,
                    'model_id': model_id,
                    'success': True
                })
            else:
                results.append({
                    'category': category,
                    'model_name': model_name,
                    'model_id': model_id,
                    'success': False
                })
    
    return pd.DataFrame(results)

# Run comparison
comparison_results = compare_models()
print("\n📊 Comparison Summary:")
print(comparison_results)

## Model Size vs Performance Analysis

Let's analyze the relationship between model size and performance.

In [ ]:
def analyze_model_sizes():
    """
    Analyze model sizes and create a comparison table.
    """
    api = HfApi()
    model_sizes = []
    
    for category, info in MODEL_CATEGORIES.items():
        for model_name, model_id in info['models'].items():
            try:
                model_info = api.model_info(model_id)
                total_size = 0
                if hasattr(model_info, 'siblings'):
                    total_size = sum(sibling.size for sibling in model_info.siblings if sibling.size) / (1024**3)  # GB
                
                model_sizes.append({
                    'category': category,
                    'model_name': model_name,
                    'model_id': model_id,
                    'size_gb': round(total_size, 2) if total_size > 0 else None,
                    'downloads': model_info.downloads if hasattr(model_info, 'downloads') else None
                })
            except Exception as e:
                model_sizes.append({
                    'category': category,
                    'model_name': model_name,
                    'model_id': model_id,
                    'size_gb': None,
                    'downloads': None
                })
    
    df = pd.DataFrame(model_sizes)
    return df.sort_values('size_gb', na_position='last')

# Analyze model sizes
size_analysis = analyze_model_sizes()
print("📏 Model Size Analysis:")
print(size_analysis)

# Create a simple visualization
try:
    import matplotlib.pyplot as plt
    
    # Filter out models without size info
    plot_data = size_analysis.dropna(subset=['size_gb'])
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(plot_data['model_name'], plot_data['size_gb'])
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Model Size (GB)')
    plt.title('HuggingFace Model Sizes by Category')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("Matplotlib not available for visualization")

## Interactive Model Explorer

Create an interactive interface to explore models.

In [ ]:
def create_model_explorer():
    """
    Create an interactive model exploration interface.
    """
    print("🤖 HuggingFace Model Explorer")
    print("=" * 40)
    
    while True:
        print("\nAvailable categories:")
        for i, (category, info) in enumerate(MODEL_CATEGORIES.items(), 1):
            print(f"{i}. {category}: {info['description']}")
        print("0. Exit")
        
        try:
            choice = input("\nSelect a category (number): ").strip()
            
            if choice == "0":
                break
                
            category_idx = int(choice) - 1
            if 0 <= category_idx < len(MODEL_CATEGORIES):
                category = list(MODEL_CATEGORIES.keys())[category_idx]
                info = MODEL_CATEGORIES[category]
                
                print(f"\n🔍 {category.upper()} Models:")
                for i, (model_name, model_id) in enumerate(info['models'].items(), 1):
                    print(f"{i}. {model_name}")
                
                model_choice = input("\nSelect a model (number): ").strip()
                model_idx = int(model_choice) - 1
                
                if 0 <= model_idx < len(info['models']):
                    model_name = list(info['models'].keys())[model_idx]
                    model_id = info['models'][model_name]
                    
                    test_prompt = input("Enter your test prompt: ").strip()
                    
                    task_type = category if category != "text_generation" else "text-generation"
                    test_model(model_id, task_type=task_type, test_prompt=test_prompt)
                else:
                    print("Invalid model selection")
            else:
                print("Invalid category selection")
                
        except ValueError:
            print("Please enter a valid number")
        except KeyboardInterrupt:
            break
    
    print("\n👋 Thanks for exploring HuggingFace models!")

# Uncomment to run interactive explorer
# create_model_explorer()

## Key Takeaways

### What We Learned:
1. **Model Variety**: HuggingFace offers models for different tasks (generation, classification, translation)
2. **Size vs Performance**: Larger models generally perform better but require more resources
3. **Task-Specific Models**: Different architectures excel at different tasks
4. **Resource Requirements**: GPU acceleration significantly speeds up inference

### Best Practices:
- Start with smaller models for experimentation
- Use GPU when available for better performance
- Choose models based on your specific use case
- Consider model size for deployment constraints

### Next Steps:
- Try fine-tuning these models on custom data
- Explore model optimization techniques
- Learn about deploying models to production